In [1]:
import gymnasium as gym
import gymnasium_env
import random

/Users/mario/Documents/proj/cam/Blokus/venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:642: UserWarning: WARN: Overriding environment gymnasium_env/Blokus-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [2]:

from tqdm import tqdm
from copy import deepcopy
import time
done = False
def encode_board(board):
    return ''.join(str(cell) for row in board for cell in row)

def decode_board(encoded_board, n):
    board = []
    for i in range(n):
        row = [int(encoded_board[i * n + j]) for j in range(n)]
        board.append(row)
    return board

def compute_optimal_score(board_size, num_players, max_depth): ## for each state, I want the optimal action
    best_action, optimal_score_v = {}, {}
    global total
    total = 0
    global different_states, function_calls, num_actions, max_actions, not_zero
    different_states, function_calls, num_actions, max_actions, not_zero = 0, 0, 0, 0, 0
    start = time.time()
    def optimal_score(state:gym.Env, depth, passed=False) -> int:
        global function_calls, different_states, num_actions, max_actions, not_zero
        function_calls += 1
        board_encoding = encode_board(state.board)
        if (board_encoding, state.current_player) in optimal_score_v:
            return optimal_score_v[board_encoding, state.current_player]
        if depth == 0:
            return 0
        different_states += 1
        possible_actions = state.possible_actions(state.current_player)
        num_actions += len(possible_actions)
        max_actions = max(max_actions, len(possible_actions))
        if len(possible_actions) == 0:
            if passed:
                return 0
            else:
                global total
                init_time = time.time()
                new_state = deepcopy(state)
                end_time = time.time()
                total += end_time - init_time
                new_state.current_player = (new_state.current_player) % new_state.num_players + 1
                return -optimal_score(new_state, depth - 1, passed=True)
        not_zero += 1
        ret = float('-inf')
        if depth == 100:
            possible_actions = tqdm(possible_actions)
        for action in possible_actions:
            init_time = time.time()
            new_state = deepcopy(state)
            end_time = time.time()
            total += end_time - init_time
            # state.render()
            obs, reward, terminated, truncated, info = new_state.step(action)
            # state.render()
            if truncated:
                assert False
            eval = optimal_score_v[encode_board(new_state.board), new_state.current_player] = optimal_score(new_state, depth - 1, terminated)
            if ret < -eval + reward:
                best_action[board_encoding, state.current_player] = action
                ret = -eval + reward
                # if depth > 97:
                #     print(ret, new_state._action_to_tuple(action), 100 - depth, different_states)
                #     print(f"Action average: {num_actions / different_states}")
                #     print(f"Max actions: {max_actions}")
                #     print(f"Average not zero{num_actions / not_zero}")
        return ret
    env = gym.make('gymnasium_env/Blokus-v0', board_size=board_size, num_players=2, render_mode='human', render_scale=10, neighborhood_dir="/Users/mario/Documents/proj/cam/Blokus/gymnasium_env/envs/auxiliary/pre_neighbors", mode="good", disable_env_checker=True)
    # env = gym.make('gymnasium_env/Blokus-v0', board_size=board_size, num_players=num_players, render_mode='human', render_scale=10, disable_env_checker=True)
    env = env.unwrapped
    env.order_enforce = False
    env.reset()
    optimal_score_v[encode_board(env.board), 1] = optimal_score(env, max_depth, False)
    end = time.time()
    print("action choosing: ", env.total_precomputed)
    print("attributes updates: ", env.total_faster)
    print("initialisation time: ", env.initialization_total_time)
    print("total copying time: ", total)
    print("other time: ", end - start - env.total_precomputed - env.total_faster - total)
    print("total time: ", end - start)
    print(total)
    return best_action, optimal_score_v, different_states, function_calls

In [3]:
BOARD_SIZE = 5

# env = gym.make('gymnasium_env/Blokus-v0', board_size=BOARD_SIZE, num_players=2, render_mode='human', render_scale=10, neighborhood_dir="/Users/mario/Documents/proj/cam/Blokus/gymnasium_env/envs/auxiliary/pre_neighbors", mode="good", disable_env_checker=True)
# env = env.unwrapped
# env.order_enforce = False

# action choosing:  0.00014591217041015625
# attributes updates:  0
# initialisation time:  3.208707094192505
# total copying time:  9.408671617507935
# other time:  4.122884511947632
# total time:  13.531702041625977
# 9.408671617507935
# different states 52679
# function calls 54861
# 15047 32080


best_action, optimal_score_v, different_states, function_calls = compute_optimal_score(board_size=BOARD_SIZE, num_players=2, max_depth=100)
print("different states", different_states)
print("function calls", function_calls)
print(len(best_action), len(optimal_score_v))

Neighborhood to encoded actions loaded
Initialised on good mode


  0%|          | 0/58 [00:51<?, ?it/s]


KeyboardInterrupt: 

In [ ]:

# env.reset()
# action1 = best_action[encode_board(env.board), 1]
# env.step(action1)
# action2 = best_action[encode_board(env.board), 2]
# env.step(action2)
# print(env.board)
# action3 = best_action[encode_board(env.board), 1]
# env.step(action3)
# action4 = best_action[encode_board(env.board), 2]
# env.step(action4)
# print(action1, action2, action3, action4)

# print(env._action_to_tuple(action1))
# print(env._action_to_tuple(action2))
# print(env._action_to_tuple(action3))
# print(env._action_to_tuple(action4))
# # print(best_action["0" * BOARD_SIZE * BOARD_SIZE, 1])
# # print(env._action_to_tuple(best_action["0" * BOARD_SIZE * BOARD_SIZE, 1]), optimal_score_v["0" * BOARD_SIZE * BOARD_SIZE, 1])
# # # print(env._action_to_tuple(best_action["1111100000000000", 2]), optimal_score_v["1111100000000000", 2])
# # # action=best_action["1111100000000000", 2]

# # print(action)


In [ ]:
action5 = best_action[encode_board(env.board), 1]
env.step(action5)
action6 = best_action[encode_board(env.board), 2]
env.step(action6)
print(action5, action6)
print(env._action_to_tuple(action5))
print(env._action_to_tuple(action6))

# import pickle

# with open('best_action.pkl', 'wb') as f:
#     pickle.dump(best_action, f, protocol=pickle.HIGHEST_PROTOCOL)


NameError: name 'env' is not defined

In [ ]:
action7 = best_action[encode_board(env.board), 1]
env.step(action7)
action8 = best_action[encode_board(env.board), 2]
env.step(action8)
print(action7, action8)
print(env._action_to_tuple(action7))
print(env._action_to_tuple(action8))

# with open('optimal_score_v.pkl', 'wb') as f:
#     pickle.dump(optimal_score_v, f, protocol=pickle.HIGHEST_PROTOCOL)